# NiN
**结构**：以一个普通卷积层开始，后面是两个$1 \times 1$的卷积层。这两个$1 \times 1$卷积层充当带有ReLU激活函数的逐像素全连接层。
第一层的卷积窗口形状通常由用户设置。
随后的卷积窗口形状固定为$1 \times 1$。 
**核心思想**：
1. 找到一个高性能的模块，随后进行组合来适应不同的数据
2. 完全放弃全连接层，转而使用1×1卷积，从而使网络变深变窄，方便计算

![对比 VGG 和 NiN 及它们的块之间主要架构差异。](../img/nin.svg)

In [ ]:
import torch
from torch import nn
from d2l import torch as d2l


def nin_block(in_channels, out_channels, kernel_size, strides, padding):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size, strides, padding),
        nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=1), nn.ReLU(),
        nn.Conv2d(out_channels, out_channels, kernel_size=1), nn.ReLU()
        )

net = nn.Sequential(
    nin_block(1, 96, kernel_size=11, strides=4, padding=0),
    nn.MaxPool2d(3, stride=2),
    nin_block(96, 256, kernel_size=5, strides=1, padding=2),
    nn.MaxPool2d(3, stride=2),
    nin_block(256, 384, kernel_size=3, strides=1, padding=1),
    nn.MaxPool2d(3, stride=2),
    nn.Dropout(0.5),
    # 标签类别数是10
    nin_block(384, 10, kernel_size=3, strides=1, padding=1),
    nn.AdaptiveAvgPool2d((1, 1)),
    # 将四维的输出转成二维的输出，其形状为(批量大小,10)
    nn.Flatten()
    )

X = torch.rand(size=(1, 1, 224, 224))
for layer in net:
    X = layer(X)
    print(layer.__class__.__name__,'output shape:\t', X.shape)

lr, num_epochs, batch_size = 0.1, 10, 128
train_iter, test_iter = d2l.load_data_fashion_mnist(batch_size, resize=224)
d2l.train_ch6(net, train_iter, test_iter, num_epochs, lr, d2l.try_gpu())

### 发现
#### 1. 参数数量 (Parameters)
**公式：** 对于卷积层 $Conv(C_{in}, C_{out}, k)$，参数量 $\approx k^2 \times C_{in} \times C_{out}$。
NiN 的核心在于去除了全连接层，参数主要集中在卷积层。

*   **NiN 块 1**: $(11^2 \cdot 1 \cdot 96) + (1^2 \cdot 96 \cdot 96) \times 2 \approx 30,048$
*   **NiN 块 2**: $(5^2 \cdot 96 \cdot 256) + (1^2 \cdot 256 \cdot 256) \times 2 \approx 745,728$
*   **NiN 块 3**: $(3^2 \cdot 256 \cdot 384) + (1^2 \cdot 384 \cdot 384) \times 2 \approx 1,179,648$
*   **NiN 块 4**: $(3^2 \cdot 384 \cdot 10) + (1^2 \cdot 10 \cdot 10) \times 2 \approx 34,780$
*   **总计**: 约 **2M**

#### 2. 计算量 (FLOPs/MACs)
**公式：** 对于每一层，$FLOPs \approx 2 \times (\text{参数量}) \times (\text{输出特征图宽} \times \text{输出特征图高})$。

由于 NiN 在每个像素上应用 MLP（$1 \times 1$ 卷积），计算量与输出分辨率高度相关。
*   第一层输出 $54 \times 54$，最后一层输出 $5 \times 5$。
*   NiN 的计算重点在早期的卷积层和 $1 \times 1$ 卷积。总计算量约为 **0.7 - 1.0 GFLOPs**（视输入尺寸 224x224 ）。

### 3. 显存占用 (VRAM)
显存由 **模型参数** 和 **中间激活值 (Activations)** 组成。

#### A. 预测期间 (Inference)
*   **模型权重**: $1.99M \times 4 \text{ bytes (float32)} \approx 8MB$。
*   **激活值**: 只需要存储当前层和下一层的中间结果。最大显存取决于最宽的特征图。
    *   例如：第一层输出 $96 \times 54 \times 54 \times 4 \text{ bytes} \approx 1.1MB$。
*   **总计**: 通常小于 **100MB**。

#### B. 训练期间 (Training)
训练显存远高于预测，因为需要存储**所有层**的激活值以计算梯度。
**估计公式：** $Memory \approx (\text{模型参数} + \text{梯度} + \text{优化器状态}) + (\text{批量大小} \times \text{单样本总激活值})$。
*   **批量大小 (Batch Size)** 是关键因素。若 $BatchSize = 128$：
    - 单样本总激活元素（按各层输出求和）= $3222384$。
    - 计算过程：$1\times224\times224 + 6\times(96\times54\times54) + 96\times26\times26 + 6\times(256\times26\times26) + 256\times12\times12 + 6\times(384\times12\times12) + 384\times5\times5 + 384\times5\times5 + 6\times(10\times5\times5) + 10 + 10 = 3222384$。
    - 单样本激活显存：$3222384 \times 4 \text{ bytes} \approx 12.9 \text{MB}$。
    - 批量激活显存：$12.9 \text{MB} \times 128 \approx 1.6 \text{GB}$（仅激活，不含参数/梯度/workspace）。
*   **优化器**: SGD 为参数量的 2 倍，Adam 为 4 倍。
*   因反向梯度、临时工作区和缓存开销，训练峰值通常约为 2.5GB - 3.5GB，实际约 3G 与估算相符。
